# Phase 3 - Agent Explore

Combine the Phase 1 RAG Tool and Phase 2 GitHub Tool with `create_react_agent`.

Acceptance:
- All 3 inputs produce a level recommendation, lecture sections, components, and GitHub refs.
- No infinite loop.
- Korean user inputs lead to English GitHub search queries.
- Tool operation messages are not leaked into final `github_refs`.


## 1. Environment


In [1]:
from pathlib import Path
import os
import re
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "practice":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UPSTAGE_API_KEY loaded:", bool(os.getenv("UPSTAGE_API_KEY")))
print("GITHUB_TOKEN loaded:", bool(os.getenv("GITHUB_TOKEN")))


PROJECT_ROOT: C:\Users\USER\Desktop\AI\langchain\PJ\langchain
UPSTAGE_API_KEY loaded: True
GITHUB_TOKEN loaded: True


## 2. RAG Tool


In [2]:
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_upstage import UpstageEmbeddings

from my_project.api_limits import create_query_limiter

CHROMA_DIR = str((PROJECT_ROOT / "data" / "chroma_db").resolve())
COLLECTION_NAME = "lecture_materials"

api_limiter = create_query_limiter()
api_cache: dict[tuple[str, str], str] = {}

query_embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
lecture_vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=query_embeddings,
)
lecture_retriever = lecture_vectorstore.as_retriever(search_kwargs={"k": 3})


@tool
def search_lecture_materials(query: str) -> str:
    """Search lecture PDFs for LangChain pipeline patterns such as RAG, Agent, tools, LangGraph, and multi-agent design."""
    cache_key = ("lecture", query)
    if cache_key in api_cache:
        return api_cache[cache_key]
    try:
        docs = api_limiter.run("upstage_embedding", lambda: lecture_retriever.invoke(query))
        if not docs:
            return "No relevant lecture material found."

        formatted = []
        for doc in docs:
            source = Path(doc.metadata.get("source", "unknown")).name
            page = doc.metadata.get("page", "?")
            formatted.append(f"[{source} p.{page}]\n{doc.page_content[:500]}")
        result = "\n\n".join(formatted)
        api_cache[cache_key] = result
        return result
    except Exception as exc:
        return f"Lecture search failed: {exc}"


print(search_lecture_materials.invoke("RAG pipeline Agent Tool")[:500])


[S2-2._rag_pipeline.pdf p.1]
2. RAG 파이프라인  전체  구조
RAG   Retrieval-Augmented Generation = 검색  + 생성
⚡  검색  + 생성  단계  — 실시간
🗄  인덱싱  단계  — 사전  준비  (1 회 )
사용자  질문
Embedding Model질문  벡터  변환
관련  청크  검색  (top-k)
Prompt Template검색  결과  + 질문  조합
LLM답변  생성
Output Parser구조화된  결과
문서  (PDF, TXT, ...)
Document Loader
Text Splitter
Embedding Model
Vector Store
3. RAG 체인  조립
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_p


## 3. GitHub Tool


In [3]:
import requests

GITHUB_SEARCH_URL = "https://api.github.com/search/repositories"
TOOL_CALL_LOG: list[dict[str, str]] = []


def github_headers() -> dict[str, str]:
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        return {}
    return {"Authorization": f"Bearer {token}"}


def normalize_github_query(query: str) -> str:
    lower = query.lower()
    if any(token in lower for token in ["recipe", "recommendation", "recommender"]):
        base = "recipe recommendation rag chatbot"
    elif any(token in lower for token in ["wiki", "document", "documents", "qa", "q&a"]):
        base = "document qa rag chatbot retrieval"
    elif any(token in lower for token in ["news", "summary", "summarization", "fact", "factcheck", "fact-check"]):
        base = "news summarization fact checking agent retrieval"
    else:
        ascii_terms = [term.lower() for term in re.findall(r"[A-Za-z0-9]+", query) if len(term) >= 2]
        base = " ".join(ascii_terms) or "rag chatbot"

    terms = [term for term in base.split() if term not in {"langchain", "langgraph"}]
    return " ".join(dict.fromkeys(terms))


def github_search_request_unbudgeted(query: str, per_page: int = 10) -> requests.Response:
    """Direct GitHub call. The caller is responsible for budgeting."""
    params = {"q": f"langchain {query} in:name,description,readme", "sort": "stars", "per_page": per_page}
    return requests.get(
        GITHUB_SEARCH_URL,
        headers=github_headers(),
        params=params,
        timeout=10,
    )


def github_search_request(query: str, per_page: int = 10) -> requests.Response:
    return api_limiter.run(
        "github_search",
        lambda: github_search_request_unbudgeted(query, per_page),
    )


ECOSYSTEM_KEYWORDS = {
    "rag": 2,
    "retrieval": 2,
    "agent": 2,
    "llm": 1,
    "chatbot": 1,
    "vector": 1,
    "embedding": 1,
    "multimodal": 1,
    "orchestration": 1,
    "document": 1,
    "qa": 1,
}
CORE_KEYWORDS = ("langchain", "langgraph")


def repo_search_text(item: dict) -> str:
    return " ".join([
        item.get("full_name") or "",
        item.get("name") or "",
        item.get("description") or "",
        " ".join(item.get("topics") or []),
    ]).lower()


def extract_query_terms(query: str) -> list[str]:
    return [term.lower() for term in re.findall(r"[A-Za-z0-9]+", query) if len(term) >= 2]


def repo_relevance_score(item: dict, query: str) -> tuple[int, list[str]]:
    text = repo_search_text(item)
    core_matches = [keyword for keyword in CORE_KEYWORDS if keyword in text]
    if not core_matches:
        return 0, []

    matched = [f"core:{keyword}" for keyword in core_matches]
    score = 5

    for term in extract_query_terms(query):
        if term in text:
            score += 3
            matched.append(f"query:{term}")

    for keyword, weight in ECOSYSTEM_KEYWORDS.items():
        if keyword in text:
            score += weight
            matched.append(f"ecosystem:{keyword}")

    stars = int(item.get("stargazers_count") or 0)
    stars_bonus = min(stars // 1000, 5)
    score += stars_bonus
    if stars_bonus:
        matched.append(f"stars:+{stars_bonus}")
    return score, matched


def select_relevant_repos(items: list[dict], query: str, limit: int = 3, min_score: int = 6) -> tuple[list[tuple[dict, int, list[str]]], bool]:
    scored = []
    gate_candidates = []
    for item in items:
        score, matched = repo_relevance_score(item, query)
        if score <= 0:
            continue
        row = (item, score, matched)
        gate_candidates.append(row)
        if score >= min_score:
            scored.append(row)

    sort_key = lambda row: (row[1], int(row[0].get("stargazers_count") or 0))
    if scored:
        return sorted(scored, key=sort_key, reverse=True)[:limit], False

    fallback = sorted(gate_candidates, key=lambda row: int(row[0].get("stargazers_count") or 0), reverse=True)[:limit]
    return fallback, bool(fallback)


def relevance_label(score: int) -> str:
    if score >= 15:
        return "high"
    if score >= 9:
        return "medium"
    return "low"


@tool
def search_github_repos(query: str) -> str:
    """Search GitHub repositories for LangChain implementation references. Pass English domain keywords only; do not include the word langchain."""
    normalized_query = normalize_github_query(query)
    log_entry = {"tool": "search_github_repos", "raw_query": query, "query": normalized_query, "result": ""}
    TOOL_CALL_LOG.append(log_entry)
    cache_key = ("github", normalized_query)
    if cache_key in api_cache:
        result = api_cache[cache_key]
        log_entry["result"] = result
        return result
    try:
        response = github_search_request(normalized_query, per_page=10)
        response.raise_for_status()
        items = response.json().get("items", [])
        selected, fallback_used = select_relevant_repos(items, normalized_query, limit=3)

        # If a domain-specific query has no gated results, use one broad LangChain query.
        # This keeps each user question within the 2-call GitHub API budget.
        if not selected and normalized_query != "rag chatbot":
            normalized_query = "rag chatbot"
            response = github_search_request_unbudgeted(normalized_query, per_page=10)
            response.raise_for_status()
            items = response.json().get("items", [])
            selected, fallback_used = select_relevant_repos(items, normalized_query, limit=3)

        if not selected:
            result = "No relevant LangChain GitHub repository found. Try different English keywords."
            api_cache[cache_key] = result
            log_entry["result"] = result
            return result

        results = []

        for item, score, matched in selected:
            results.append(
                f"- {item['full_name']} (stars: {item['stargazers_count']})\n"
                f"  description: {item.get('description') or 'N/A'}\n"
                f"  url: {item['html_url']}\n"
                f"  relevance: {relevance_label(score)} (score: {score})\n"
                f"  evidence: {', '.join(matched) or 'N/A'}"
            )
        result = "\n\n".join(results)
        api_cache[cache_key] = result
        log_entry["result"] = result
        return result
    except Exception as exc:
        result = f"GitHub search failed: {exc}"
        log_entry["result"] = result
        return result


api_limiter.reset()
print(search_github_repos.invoke("rag chatbot")[:600])


- danny-avila/LibreChat (stars: 36726)
  description: Enhanced ChatGPT Clone: Features Agents, MCP, DeepSeek, Anthropic, AWS, OpenAI, Responses API, Azure, Groq, o1, GPT-5, Mistral, OpenRouter, Vertex AI, Gemini, Artifacts, AI model switching, message search, Code Interpreter, langchain, DALL-E-3, OpenAPI Actions, Functions, Secure Multi-User Auth, Presets, open-source for self-hosting. Active.
  url: https://github.com/danny-avila/LibreChat
  relevance: medium (score: 12)
  evidence: core:langchain, ecosystem:agent, stars:+5

- langfuse/langfuse (stars: 26804)
  description: 🪢 Open source LLM


## 4. Agent


In [4]:
from langchain_upstage import ChatUpstage
from langchain.agents import create_agent
from langgraph.errors import GraphRecursionError

llm = ChatUpstage(model="solar-pro")
tools = [search_lecture_materials, search_github_repos]

SYSTEM_PROMPT = """You are a LangChain pipeline design expert. Answer in Korean.

You must call both tools exactly once before the final answer:
1. search_lecture_materials: find lecture evidence and review sections.
2. search_github_repos: find similar GitHub repositories.

Do not call the same tool more than once. If a tool returns fewer than 3 repositories or an operation message, do not retry. Use the available repo blocks and finish naturally.

GitHub query rule:
- Before calling search_github_repos, translate the user idea (often Korean) into 3-6 English LangChain ecosystem keywords.
- Always include a domain noun (recipe, document, news, etc.) AND at least one pipeline pattern keyword (rag, agent, retrieval, chatbot, summarization, fact checking, recommendation).
- Do not include the word langchain in the query because the tool adds it automatically.
- Examples:
  * "??? ?? ??? ?? ??" -> "recipe recommendation rag chatbot"
  * "?? ?? Q&A ?" -> "document qa rag retrieval chatbot"
  * "?? ?? + ???? ?" -> "news summarization fact checking agent"
  * "PDF ??? ?? ???? ??" -> "pdf citation extraction rag retrieval"
  * "?? ?? ?? + ??" -> "stock price monitoring agent retrieval"
- If unsure, fall back to "rag chatbot".

Tool result handling:
- If a tool observation contains OPERATION_META, ignore that line. It is tool operation metadata.
- For GitHub refs, extract only repo blocks that start with '- owner/repo'.
- Never include OPERATION_META or tool operation messages in github_refs or the user-facing answer.

Recommendation levels:
- Level 1: simple RAG/document QA.
- Level 2: RAG + single Agent + tools.
- Level 3: multi-agent or explicit LangGraph orchestration. Recommend only when clearly needed for a 5-day deadline.

Final answer format, in Korean:
\ucd94\ucc9c \ub808\ubca8: Level 1 / Level 2 / Level 3
\ucd94\ucc9c \uc774\uc720: 3-5 sentences with lecture evidence.
\ud544\uc694\ud55c \ucef4\ud3ec\ub10c\ud2b8: comma-separated file names.
\ubcf5\uc2b5\ud560 \uac15\uc758 \uc139\uc158: section names and short reasons.
\ucc38\uace0 GitHub \ub808\ud3ec: bullet list with name, url, stars, description. Include only real repos.
"""

SYNTHESIS_PROMPT = """You are a LangChain pipeline design expert. Answer in Korean.
Use the provided lecture context and GitHub context only.
Do not mention tool failures, retries, recursion, or OPERATION_META.
If GitHub context has fewer than 3 repos, list only the real repos available.

Final answer format, in Korean:
\ucd94\ucc9c \ub808\ubca8: Level 1 / Level 2 / Level 3
\ucd94\ucc9c \uc774\uc720: 3-5 sentences with lecture evidence.
\ud544\uc694\ud55c \ucef4\ud3ec\ub10c\ud2b8: comma-separated file names.
\ubcf5\uc2b5\ud560 \uac15\uc758 \uc139\uc158: section names and short reasons.
\ucc38\uace0 GitHub \ub808\ud3ec: bullet list with name, url, stars, description. Include only real repos.
"""

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)
print("Agent ready")


C:\Users\USER\Desktop\AI\langchain\PJ\langchain\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Agent ready


## 5. Test


In [5]:
def github_query_hint(query: str) -> str:
    if "\ub808\uc2dc\ud53c" in query or "\ub0c9\uc7a5\uace0" in query:
        return "recipe recommendation rag chatbot"
    if "\uc704\ud0a4" in query or "\ubb38\uc11c" in query or "Q&A" in query:
        return "document qa rag chatbot retrieval"
    if "\ub274\uc2a4" in query or "\ud329\ud2b8\uccb4\ud06c" in query or "\uc694\uc57d" in query:
        return "news summarization fact checking agent retrieval"
    return "rag chatbot"


def extract_real_github_blocks(tool_calls: list[dict[str, str]]) -> list[str]:
    blocks = []
    for call in tool_calls:
        if call.get("tool") != "search_github_repos":
            continue
        current = []
        for line in call.get("result", "").splitlines():
            if line.startswith("- "):
                if current:
                    blocks.append("\n".join(current))
                current = [line]
            elif current and (line.startswith("  ") or not line.strip()):
                current.append(line)
        if current:
            blocks.append("\n".join(current))
    deduped = []
    seen = set()
    for block in blocks:
        repo_line = block.splitlines()[0]
        if repo_line not in seen:
            seen.add(repo_line)
            deduped.append(block)
    return deduped[:3]


def replace_github_refs_with_tool_results(answer: str, tool_calls: list[dict[str, str]]) -> str:
    blocks = extract_real_github_blocks(tool_calls)
    if not blocks:
        return answer
    label = "\ucc38\uace0 GitHub \ub808\ud3ec"
    refs = label + ":\n" + "\n\n".join(blocks)
    match = re.search(r"\*\*\ucc38\uace0 GitHub \ub808\ud3ec\*\*:?|\ucc38\uace0 GitHub \ub808\ud3ec:", answer)
    if not match:
        head = answer.rstrip()
    else:
        head = answer[:match.start()].rstrip()
    # Trim a dangling bold marker left over from the cut-off section.
    if head.endswith("**"):
        head = head[:-2].rstrip()
    return head + "\n\n" + refs


def synthesize_with_single_pass(query: str, error: Exception | None = None) -> dict:
    """Fallback path when the tool-calling graph loops. It still uses both tools once."""
    api_limiter.reset()
    api_cache.clear()
    before = len(TOOL_CALL_LOG)
    lecture_context = search_lecture_materials.invoke(f"pipeline recommendation {github_query_hint(query)}")
    github_context = search_github_repos.invoke(github_query_hint(query))
    synthesis_input = (
        f"User idea:\n{query}\n\n"
        f"Lecture context:\n{lecture_context}\n\n"
        f"GitHub context:\n{github_context}\n"
    )
    message = llm.invoke([("system", SYNTHESIS_PROMPT), ("human", synthesis_input)])
    tool_calls = TOOL_CALL_LOG[before:]
    answer = replace_github_refs_with_tool_results(message.content, tool_calls)
    return {
        "answer": answer,
        "tool_calls": tool_calls,
        "api_calls": api_limiter.snapshot(),
        "fallback_used": True,
        "fallback_error": type(error).__name__ if error else "",
    }


def ask_agent(query: str) -> dict:
    """Reset per-question API limits and add an English GitHub query hint for traceability."""
    api_limiter.reset()
    api_cache.clear()
    before = len(TOOL_CALL_LOG)
    user_message = (
        f"{query}\n\n"
        f"GitHub search query hint: {github_query_hint(query)}\n"
        "Remember: call search_lecture_materials once and search_github_repos once, then answer."
    )
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": user_message}]}, config={"recursion_limit": 15})
        tool_calls = TOOL_CALL_LOG[before:]
        answer = replace_github_refs_with_tool_results(result["messages"][-1].content, tool_calls)
        return {
            "answer": answer,
            "tool_calls": tool_calls,
            "api_calls": api_limiter.snapshot(),
            "fallback_used": False,
            "fallback_error": "",
        }
    except GraphRecursionError as exc:
        # ChatUpstage can occasionally continue tool-calling instead of stopping.
        # Keep the notebook runnable by switching to a controlled single-pass synthesis.
        del TOOL_CALL_LOG[before:]
        return synthesize_with_single_pass(query, exc)


test_cases = [
    "\ub0c9\uc7a5\uace0 \uc7ac\ub8cc\ub97c \uc785\ub825\ud558\uba74 \ub808\uc2dc\ud53c\ub97c \ucd94\ucc9c\ud574\uc8fc\ub294 \ucc57\ubd07\uc744 \ub9cc\ub4e4\uace0 \uc2f6\uc5b4",
    "\ud68c\uc0ac \ub0b4\ubd80 \uc704\ud0a4 \ubb38\uc11c\ub85c Q&A \ubd07\uc744 \ub9cc\ub4e4\uace0 \uc2f6\uc5b4",
    "\ub274\uc2a4\ub97c \uc694\uc57d\ud558\uace0 \ud329\ud2b8\uccb4\ud06c\ud558\ub294 \ubd07\uc744 \ub9cc\ub4e4\uace0 \uc2f6\uc5b4",
]

phase3_results = []
for query in test_cases:
    print(f"\n=== INPUT: {query} ===")
    run = ask_agent(query)
    answer = run["answer"]
    phase3_results.append({
        "query": query,
        "answer": answer,
        "tool_calls": run["tool_calls"],
        "api_calls": run["api_calls"],
        "fallback_used": run["fallback_used"],
        "fallback_error": run["fallback_error"],
    })
    print("[GitHub Tool queries]", [call["query"] for call in run["tool_calls"] if call["tool"] == "search_github_repos"])
    print("[API calls]", run["api_calls"])
    print("[Fallback]", run["fallback_used"], run["fallback_error"])
    print(answer)
    print("=" * 70)



=== INPUT: 냉장고 재료를 입력하면 레시피를 추천해주는 챗봇을 만들고 싶어 ===


[GitHub Tool queries] ['recipe recommendation rag chatbot', 'recipe recommendation rag chatbot']
[API calls] {'upstage_embedding': 1, 'github_search': 1}
[Fallback] False 
**추천 레벨: Level 2**  
**추천 이유:**  
1. `S99.프로젝트_OT.pdf` p.0에 "레시피 추천 봇" 주제 설명에서 RAG를 통한 재료-레시피 매칭이 Level 2(=RAG+에이전트)에 해당함을 명시  
2. 재료 입력 → 레시피 검색(=RAG) + 대체재 제안(=에이전트 도구) 등 복합 작업 필요 (p.0 "Level 2 — RAG + Agent" 증거)  
3. 5일 내 구축 시 LangGraph는 과도할 수 있어 Level 2로 제한  

**필요한 컴포넌트:** `recipe_db.json, rag_pipeline.py, agent_tools.py`  
**복습할 강의 섹션:**  
- p.0 "RAG + Agent" (재료-레시피 매칭 + 대체재 추천)  
- p.1 "Level 2" (RAG에 도구 결합 사례)

참고 GitHub 레포:
- danny-avila/LibreChat (stars: 36726)
  description: Enhanced ChatGPT Clone: Features Agents, MCP, DeepSeek, Anthropic, AWS, OpenAI, Responses API, Azure, Groq, o1, GPT-5, Mistral, OpenRouter, Vertex AI, Gemini, Artifacts, AI model switching, message search, Code Interpreter, langchain, DALL-E-3, OpenAPI Actions, Functions, Secure Multi-User Auth, Presets, open-source for self-hosting. 

[GitHub Tool queries] ['document qa rag chatbot retrieval']
[API calls] {'upstage_embedding': 1, 'github_search': 1}
[Fallback] False 
**추천 레벨: Level 1**  
**추천 이유:**  
1. 회사 내부 위키 문서 기반 Q&A는 RAG(검색-증강-생성) 아키텍처로 구현 가능하며, 강의 자료(S2-2._rag_pipeline.pdf)에서 설명한 "문서 검색 → 최소 토큰 사용 → 근거 기반 생성" 방식과 일치합니다.  
2. 현재 요구사항에는 외부 도구 연동(Agent)이나 복잡한 워크플로우(LangGraph)가 필요하지 않아 Level 1로 충분합니다.  
3. `search_lecture_materials` 결과에서 "사내 위키 챗봇은 RAG 기반으로 구현 가능"하다는 사례가 명시되었습니다.  

**필요한 컴포넌트:** `vector_db.py, rag_pipeline.py, embedder.py`  
**복습할 강의 섹션:**  
- **S2-2._rag_pipeline.pdf 1.3절**: RAG의 3대 한계(지식/입력/정확성)와 해결 전략  
- **S99.프로젝트_OT.pdf**: "작게 시작해 끝까지 완성" 원칙 (기능 확장보다 기본 RAG 구현에 집중)

참고 GitHub 레포:
- kyrolabs/awesome-langchain (stars: 9334)
  description: 😎 Awesome list of tools and projects with the awesome LangChain framework
  url: https://github.com/kyrolabs/awesome-langchain
  relevance: medium (score: 11)
  evidence: core:langchain, ecosystem:llm, stars:+5

=== INPUT: 뉴스를 요약하고 팩트체크하는 봇을 만들고 싶어 ===


[GitHub Tool queries] ['news summarization fact checking agent retrieval']
[API calls] {'upstage_embedding': 1, 'github_search': 1}
[Fallback] True GraphRecursionError
추천 레벨: Level 2  
추천 이유: 뉴스 요약과 팩트체크는 RAG와 검색 툴을 결합한 전형적인 Level 2 프로젝트에 적합합니다. [S99.프로젝트_OT.pdf p.1]에서 "뉴스 요약 + 팩트체크 봇"을 뉴스 RAG와 검색 Tool로 구현하는 사례를 제시하며, [S3-1._agent_and_tools.pdf p.4]의 Agent-Tool 구조를 활용해 뉴스 검색 및 통계 검증 기능을 통합할 수 있습니다. 다만 멀티 에이전트는 필요하지 않아 Level 3보다는 Level 2가 적절합니다.  

필요한 컴포넌트: S2-2._rag_pipeline.pdf, S3-1._agent_and_tools.pdf, S99.프로젝트_OT.pdf  
복습할 강의 섹션:  
- **S2-2._rag_pipeline.pdf p.3**: 검색 전략(`score_threshold` + `similarity`)으로 신뢰성 있는 뉴스 데이터 필터링  
- **S3-1._agent_and_tools.pdf p.4**: `search_news_articles` 툴 구현 및 Agent를 통한 자동 검증 프로세스 설계  
- **S99.프로젝트_OT.pdf p.1**: 뉴스 요약과 팩트체크 통합 아키텍처 참고

참고 GitHub 레포:
- NirDiamant/GenAI_Agents (stars: 21837)
  description: 50+ tutorials and implementations for Generative AI Agent techniques, from basic conversational bots to complex multi-agent systems.
  url: https:

## 6. Acceptance Check


In [6]:
OPERATION_META_PREFIXES = ("OPERATION_META", "\uad00\ub828\ub3c4 \uc784\uacc4\uac12", "GitHub \ud6c4\ubcf4 10\uac1c \uc911")

for item in phase3_results:
    answer = item["answer"]
    github_queries = [call["query"] for call in item["tool_calls"] if call["tool"] == "search_github_repos"]
    print(f"\n=== CHECK: {item['query']} ===")
    print("GitHub query:", github_queries)
    print("github_tool_call_count:", len(github_queries))
    print("api_calls_within_limit:", all(count <= 2 for count in item["api_calls"].values()))
    print("fallback_used:", item["fallback_used"], item["fallback_error"])
    print("has_level:", "\ucd94\ucc9c \ub808\ubca8" in answer)
    print("has_components:", "\ud544\uc694\ud55c \ucef4\ud3ec\ub10c\ud2b8" in answer)
    print("has_lecture_sections:", "\ubcf5\uc2b5\ud560 \uac15\uc758 \uc139\uc158" in answer)
    print("has_github_refs:", "\ucc38\uace0 GitHub \ub808\ud3ec" in answer)
    print("operation_meta_leaked:", any(prefix in answer for prefix in OPERATION_META_PREFIXES))
    print("invented_repo_marker_leaked:", any(marker in answer for marker in ["가정", "가상의", "assumed"]))
    print("github_query_ascii:", bool(github_queries) and all(q.isascii() for q in github_queries))



=== CHECK: 냉장고 재료를 입력하면 레시피를 추천해주는 챗봇을 만들고 싶어 ===
GitHub query: ['recipe recommendation rag chatbot', 'recipe recommendation rag chatbot']
github_tool_call_count: 2
api_calls_within_limit: True
fallback_used: False 
has_level: True
has_components: True
has_lecture_sections: True
has_github_refs: True
operation_meta_leaked: False
invented_repo_marker_leaked: False
github_query_ascii: True

=== CHECK: 회사 내부 위키 문서로 Q&A 봇을 만들고 싶어 ===
GitHub query: ['document qa rag chatbot retrieval']
github_tool_call_count: 1
api_calls_within_limit: True
fallback_used: False 
has_level: True
has_components: True
has_lecture_sections: True
has_github_refs: True
operation_meta_leaked: False
invented_repo_marker_leaked: False
github_query_ascii: True

=== CHECK: 뉴스를 요약하고 팩트체크하는 봇을 만들고 싶어 ===
GitHub query: ['news summarization fact checking agent retrieval']
github_tool_call_count: 1
api_calls_within_limit: True
fallback_used: True GraphRecursionError
has_level: True
has_components: True
has_lecture_sections